# Customer Catalogue GP Homologation

## Step 1 - Load, validate and normalize Global Parents

This notebook:

- Loads the customer catalogue
- Validates required columns
- Counts unique Ship To Numbers per Global Parent
- Creates a normalized GP name
- Exports a summary for review

The original catalogue is NEVER modified.

In [8]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

In [9]:
# ==========================================
# Configuration
# ==========================================

INPUT_FILE = r'C:\Users\Ivan.DeLaFuente\OneDrive - Quaker Houghton\Documents\QH WorkDay\July 2026\Industrial Market\Industrial_Market.csv'

OUTPUT_FOLDER = "output"

GP_COLUMN = "Global Parent"

SHIP_TO_COLUMN = "Ship To Number"

REMOVE_ACCENTS = True
REMOVE_PUNCTUATION = True
REMOVE_EXTRA_SPACES = True
REMOVE_LEGAL_SUFFIXES = True

In [10]:
LEGAL_SUFFIXES = [

    "INCORPORATED",
    "INC",
    "CORPORATION",
    "CORP",
    "COMPANY",
    "CO",
    "LIMITED",
    "LTD",
    "LLC",
    "LLP",
    "PLC",
    "LP",
    "GMBH",
    "AG",
    "BV",
    "NV",
    "SA",
    "SAS",
    "SPA",
    "SRL",
    "PTY",
    "PVT",
    "KK"

]

In [11]:
def normalize_gp(name):

    if pd.isna(name):
        return ""

    text = str(name).upper()

    if REMOVE_ACCENTS:
        text = unicodedata.normalize("NFKD", text)
        text = "".join(c for c in text if not unicodedata.combining(c))

    if REMOVE_PUNCTUATION:
        text = re.sub(r"[^\w\s]", " ", text)

    if REMOVE_LEGAL_SUFFIXES:

        words = text.split()

        words = [
            word
            for word in words
            if word not in LEGAL_SUFFIXES
        ]

        text = " ".join(words)

    if REMOVE_EXTRA_SPACES:
        text = re.sub(r"\s+", " ", text).strip()

    return text

In [12]:
df = pd.read_csv(INPUT_FILE)

print(f"Rows loaded: {len(df):,}")

Rows loaded: 49,493


In [13]:
required_columns = [

    GP_COLUMN,
    SHIP_TO_COLUMN

]

missing = [

    column
    for column in required_columns
    if column not in df.columns

]

if missing:

    raise ValueError(
        f"Missing columns: {missing}"
    )

print("All required columns found.")

All required columns found.


In [ ]:
df["Normalized GP"] = df[GP_COLUMN].apply(normalize_gp)

df["Normalization Changed"] = (
    df[GP_COLUMN].str.upper().fillna("")
    !=
    df["Normalized GP"]
)

In [ ]:
gp_summary = (

    df.groupby(GP_COLUMN)
      .agg(

          Ship_To_Count=(
              SHIP_TO_COLUMN,
              "nunique"
          ),

          Row_Count=(
              SHIP_TO_COLUMN,
              "count"
          ),

          GP_Country=(
              "GP Country",
              "first"
          ),

          Normalization_Changed=(
              "Normalization Changed",
              "first")

      )

      .reset_index()

)

In [16]:
gp_summary = gp_summary.sort_values(

    by=[
        "Ship_To_Count",
        GP_COLUMN
    ],

    ascending=[
        False,
        True
    ]

)

In [17]:
gp_summary.head(20)

,Global Parent,Ship_To_Count,Row_Count,GP_Country,Normalized_GP
16887,TOTAL,121,272,France,TOTAL
14566,SAMPLE ACCOUNT,105,185,NaN,SAMPLE ACCOUNT
316,ADJUSTMENT,94,202,England,ADJUSTMENT
5561,FASTENAL,94,178,United States Of America,FASTENAL
12565,PARKER HANNIFIN,87,151,United States Of America,PARKER HANNIFIN
9978,Linde,76,145,Ireland,LINDE
2078,BODYCOTE,70,131,England,BODYCOTE
4693,EATON,66,133,United States Of America,EATON
12693,PENINSULAR DE LUBRICANTES,65,70,Spain,PENINSULAR DE LUBRICANTES
3967,DANFOSS,50,100,Denmark,DANFOSS


In [18]:
Path(OUTPUT_FOLDER).mkdir(exist_ok=True)

output_file = Path(OUTPUT_FOLDER) / "01_gp_summary.csv"

gp_summary.to_csv(

    output_file,

    index=False

)

print(f"Saved to: {output_file}")

Saved to: output\01_gp_summary.csv


In [ ]:
exact_groups = (
    gp_summary
    .groupby("Normalized_GP")
    .agg(
        Variants=(GP_COLUMN, list),
        Number_of_Variants=(GP_COLUMN, "nunique"),
        Total_Ship_To_Count=("Ship_To_Count", "sum"),
        Total_Row_Count=("Row_Count", "sum"),
        Countries=("GP_Country", lambda x: sorted(set(x.dropna()))),
        Any_Normalization_Changed=("Normalization_Changed", "any")
    )
    .reset_index()
)

In [ ]:
exact_groups = exact_groups[
    exact_groups["Number_of_Variants"] > 1
].copy()

In [ ]:
exact_groups = exact_groups.sort_values(
    by=[
        "Total_Ship_To_Count",
        "Number_of_Variants"
    ],
    ascending=False
)

In [ ]:
exact_groups.head(20)

In [ ]:
output_file = Path(OUTPUT_FOLDER) / "02_exact_normalized_groups.csv"

exact_groups.to_csv(
    output_file,
    index=False
)

print(f"Saved to: {output_file}")